In [178]:
import numpy as np 
import pandas as pd 

In [179]:
df = pd.DataFrame([[8,8,4],[7, 9, 5],[6, 10, 6], [5, 12, 7]], columns=['cgpa', 'profile_score', 'lpa'])

In [180]:
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


In [181]:
def initialize_parameters(layers_dims):
    np.random.seed(3)
    parameters = {}
    L = len(layers_dims)
    for l in range(1, L):
        parameters['w' +  str(l)] = np.ones((layers_dims[l-1], layers_dims[l]))*0.1
        parameters['b' +  str(l)] = np.zeros((layers_dims[l], 1))

    return parameters

In [182]:
p = initialize_parameters([2,2,1])

In [183]:
print(p)
print(len(p))
print(len(p)//2)
print(list(i for i in range(1, (len(p)//2 + 1))))

{'w1': array([[0.1, 0.1],
       [0.1, 0.1]]), 'b1': array([[0.],
       [0.]]), 'w2': array([[0.1],
       [0.1]]), 'b2': array([[0.]])}
4
2
[1, 2]


In [184]:
def linear_forward(o_per, w, b):
    # Transpose w so the dimensions align properly with column vectors (x)
    o = np.dot(w.T, o_per) + b
    return o

In [185]:
def layer_forward(X, params):
    x = X.copy()
    l = len(params) // 2
    x_prev = 0
    layers_outputs = {}
    for i in range(1, l + 1):

        # print('-'*20)
        # print('layer: ', i)
        
        # print('x: ', x)
        # print('x_prev: ', x_prev)
        x_prev = x
        wl = params['w' + str(i)]
        bl = params['b' + str(i)]
        
        # print('-'*10)
        # print('w' + str(i) + ': ', wl)
        # print('b' + str(i) + ': ', bl)
        # print('-'*10)
        x = linear_forward(x_prev, wl, bl)
        layers_outputs['x' + str(i)] = x
        # print('layer result: ', x)
        # print('-'*10)
        # print('-'*20)
    return layers_outputs

    

In [186]:
# Convert the features dataframe to a numpy array (shape: 4, 2)
X_features = df.drop(columns=['lpa']).to_numpy()

# Loop through each student's data row-by-row
for index, row in enumerate(X_features):
    print(f"\n=== Processing Student {index + 1} ===")
    # Reshape each row from (2,) to a column vector of shape (2, 1)
    x = row.reshape(2, 1)
    layer_forward(x, p)


=== Processing Student 1 ===

=== Processing Student 2 ===

=== Processing Student 3 ===

=== Processing Student 4 ===


In [187]:
# Grab the first row, convert to numpy, and reshape to (2, 1)
x = df.drop(columns=['lpa']).iloc[0].to_numpy().reshape(2, 1)

y = df['lpa']
lo = layer_forward(x, p)

print(lo)

{'x1': array([[1.6],
       [1.6]]), 'x2': array([[0.32]])}


In [188]:
# def update_params(params, layers_output, y, y_hat, learning_rate, struct):
#     l = len(struct) - 1
#     d_loss = learning_rate * 2 *(y - y_hat)
#     for i in range(1, l+1):
#         for j in range(0, struct[i]):
#             for k in range(0, struct[i-1]):
#                 params['w'+str(i)][k][j] = params['w'+str(i)][k][j] + (d_loss * layers_output[k][0])
#             params['b'+str(i)][0][j] = params['b'+str(i)][0][j] + (d_loss)
              

# def update_params(params, layers_output, y, y_hat, learning_rate, struct):
#     l = len(struct) - 1
#     d_loss = learning_rate * 2 * (y - y_hat)
    
#     for i in range(1, l + 1):
#         # Grab the activation outputs of the PREVIOUS layer using capital 'X'
#         prev_layer_output = layers_output['x' + str(i - 1)]
        
#         for j in range(0, struct[i]):
#             for k in range(0, struct[i-1]):
#                 # Use prev_layer_output instead of integer k
#                 params['w' + str(i)][k][j] = params['w' + str(i)][k][j] + (d_loss * prev_layer_output[k][0])
                
#             params['b' + str(i)][j][0] = params['b' + str(i)][j][0] + d_loss



def update_params(params, layers_output, y, y_hat, learning_rate, struct):
    l = len(struct) - 1
    d_loss = learning_rate * 2 * (y - y_hat)
    
    for i in range(1, l + 1):
        # Grab the activation outputs of the PREVIOUS layer using capital 'X'
        prev_layer_output = layers_output['x' + str(i - 1)]
        
        for j in range(0, struct[i]):
                # Use prev_layer_output instead of integer k
            params['w' + str(i)][:, j] = params['w' + str(i)][: , j] + (d_loss * prev_layer_output[:, 0])    
            params['b' + str(i)][j][0] = params['b' + str(i)][j][0] + d_loss

In [189]:
# 1. Define the network structure matching [2, 2, 1]
print(p)
struct = [2, 2, 1]

# 2. Grab the first student's features and target value
x = df.drop(columns=['lpa']).iloc[0].to_numpy().reshape(2, 1)
y_actual = df['lpa'].iloc[0] # Target value (e.g., 4)

# 3. Run the forward pass
layers_outputs = layer_forward(x, p)

# FIX: Use capital 'X' to match layers_outputs keys ('X1', 'X2')
y_hat = layers_outputs['x' + str(len(struct) - 1)][0][0] 

print(f"Initial y_hat: {y_hat}")
print(f"Actual y: {y_actual}")

# 4. Inject 'X0' so layer 1 can access the original input vector x during updates
layers_outputs['x0'] = x

# 5. Run the update function
update_params(
    params=p, 
    layers_output=layers_outputs, 
    y=y_actual, 
    y_hat=y_hat, 
    learning_rate=0.01, 
    struct=struct
)

print("\nParameters updated successfully!")

print(p)

{'w1': array([[0.1, 0.1],
       [0.1, 0.1]]), 'b1': array([[0.],
       [0.]]), 'w2': array([[0.1],
       [0.1]]), 'b2': array([[0.]])}
Initial y_hat: 0.32000000000000006
Actual y: 4

Parameters updated successfully!
{'w1': array([[0.6888, 0.6888],
       [0.6888, 0.6888]]), 'b1': array([[0.0736],
       [0.0736]]), 'w2': array([[0.21776],
       [0.21776]]), 'b2': array([[0.0736]])}
